#1. Basic Tasks

##1. Use CTAS with read_files() to ingest a CSV file into a managed Delta table. 

In [0]:
create or replace table cyntexa_dev.sales.sales_raw as
select * from read_files('/Volumes/dev/demo/raw/sales/',format => 'csv',header => true)

##2. Ingest a nested JSON file, extracting at least 2 nested fields into top-level columns.

In [0]:
INSERT INTO cyntexa_dev.sales.sales_raw
SELECT 
  customer_id,
  order_exploded.order_id,
  order_exploded.transaction_id,
  order_exploded.product_id,
  order_exploded.quantity,
  order_exploded.amount.discount AS discount_amount,
  order_exploded.amount.total AS total_amount,
  order_exploded.order_date,
  _rescued_data
FROM read_files('/Volumes/cyntexa_dev/sales/external_data/sales_nested.json', format => 'json', multiLine => 'true')
CROSS JOIN LATERAL EXPLODE(orders) AS t(order_exploded)

###3. Run DESCRIBE, DESCRIBE EXTENDED, and DESCRIBE DETAIL on your new table and note what unique information each one gives you. 

In [0]:
DESCRIBE cyntexa_dev.sales.sales_raw;
DESCRIBE EXTENDED cyntexa_dev.sales.sales_raw;
DESCRIBE DETAIL cyntexa_dev.sales.sales_raw

`DESCRIBE:`
- We use describe to inspect and display the structural metadata of a database table or view.

`DESCRIBE EXTENDED:`
- used to retrieve detailed metadata about database objects, Basic Metadata, Physical Storage asnd Optimizations.

`DESCRIBE DETAIL:`
- Use `DESCRIBE DETAIL` to retrieve detailed metadata about a Delta Lake or Apache Iceberg table, including file count, data size, partition columns, and enabled table features.

#2. Intermediate Tasks 

##4. Add _metadata.file_name and _metadata.file_path to your ingestion query and use them to prove which source file each row came from. 

In [0]:
CREATE or REPLACE table cyntexa_dev.sales.sales_raw_metadata AS SELECT 
*,_metadata.file_name,_metadata.file_path from read_files('/Volumes/dev/demo/raw/sales/');
select * from cyntexa_dev.sales.sales_raw_metadata

##5. Create an Iceberg table from the same source data and compare its DESCRIBE DETAIL output (format, location) to the Delta version. 

In [0]:
create or replace table cyntexa_dev.sales.sales_raw_iceberg using iceberg as select * from read_files('/Volumes/dev/demo/raw/sales/');
describe detail cyntexa_dev.sales.sales_raw;
describe detail cyntexa_dev.sales.sales_raw_iceberg;

## Comparison: Delta vs Iceberg DESCRIBE DETAIL Output

**Format:**
- Delta table format: `delta`
- Iceberg table format: `iceberg`

**Location:**
- Both tables are stored in Unity Catalog managed locations
- The actual paths differ based on the table format's metadata structure:
  - Delta uses transaction log (`_delta_log/`) for metadata
  - Iceberg uses metadata files and manifests for tracking data files

##6. (Data Analyst) Write a query using the metadata columns to build a 'records per source file' audit report — useful for verifying a vendor's daily file drop. 

In [0]:
SELECT
    file_name AS source_file_name,
    COUNT(*) AS total_record_count,
    CURRENT_TIMESTAMP() AS audit_executed_at
FROM 
    cyntexa_dev.sales.sales_raw_metadata
GROUP BY 
   file_name
ORDER BY 
    source_file_name DESC;


#3. Advanced Tasks 

##7. Land multiple CSV files with slightly different formats (e.g., an extra column, a different delimiter) and document how your read_files() options need to change for each, plus how you'd detect a mismatch before it silently breaks downstream reports. 

In [0]:
CREATE OR REPLACE TABLE cyntexa_dev.sales.sales_raw_combined AS
SELECT *, _metadata.file_name AS source_file, _metadata.file_path AS source_path
FROM read_files(
  '/Volumes/cyntexa_dev/sales/external_data/sales4.csv',
  format => 'csv', header => true, delimiter => ',',
  schema => 'order_id INT, customer_id INT, transaction_id INT, product_id INT, quantity INT, discount_amount DOUBLE, total_amount DOUBLE, order_date STRING, _rescued_data STRING',rescuedDataColumn=>'_rescued_data'
)

UNION ALL

SELECT *, _metadata.file_name, _metadata.file_path
FROM read_files(
  '/Volumes/cyntexa_dev/sales/external_data/sales5.csv',
  format => 'csv', header => true, delimiter => ';',
  schema => 'order_id INT, customer_id INT, transaction_id INT, product_id INT, quantity INT, discount_amount DOUBLE, total_amount DOUBLE, order_date STRING, _rescued_data STRING',rescuedDataColumn=>'_rescued_data'
)

UNION ALL

SELECT *, _metadata.file_name, _metadata.file_path
FROM read_files(
  '/Volumes/cyntexa_dev/sales/external_data/sales6.csv',
  format => 'csv', header => true, delimiter => '\t',
  schema => 'order_id INT, customer_id INT, transaction_id INT, product_id INT, quantity INT, discount_amount DOUBLE, total_amount DOUBLE, order_date STRING, _rescued_data STRING',rescuedDataColumn=>'_rescued_data'
);


In [0]:
SELECT source_file, COUNT(*) AS rescued_row_count
FROM cyntexa_dev.sales.sales_raw_combined
WHERE _rescued_data IS NOT NULL
GROUP BY source_file;


### Detecting Schema Mismatches Before They Break Downstream Reports
**Use `_rescued_data` column** - Captures data that doesn't match the expected schema. Monitor for non-null values:
   ```sql
   SELECT source_file, COUNT(*) AS rescued_row_count
   FROM table WHERE _rescued_data IS NOT NULL
   GROUP BY source_file;
   ```


##8. Write a decision memo: when should Cyntexa choose Iceberg (or Delta UniForm) over native Delta for a given table, considering downstream tools like Snowflake or Trino? 




###**Iceberg vs. Delta Lake vs. Delta UniForm**
##### **Choose Native Delta Lake When:**

1. **Databricks-Only Environment**
   - All data consumers use Databricks (notebooks, SQL, jobs, dashboards)
   - No external query engines need direct table access
   - Full access to Delta-exclusive features: Liquid Clustering, Predictive Optimization, Change Data Feed (CDF)

2. **Maximum Performance Requirements**
   - Need the fastest possible query performance and optimization
   - Benefit from Photon acceleration and automatic optimizations
   - Require advanced Delta features like Z-ordering or column statistics
---

##### **Choose Delta UniForm When:**

1. **Multi-Engine Read Access Required**
   - Downstream tools need direct table access: Snowflake, Trino, Presto, Athena, Starburst, Dremio
   - Want to maintain a single source of truth without data duplication
   - Need Iceberg compatibility **without sacrificing Delta features**

2. **Hybrid Analytics Architecture**
   - Want to preserve Delta Lake's performance and features while enabling Iceberg reads
   - Avoid ETL/replication overhead between systems
---
##### **Choose Native Iceberg When:**

1. **Iceberg-Specific Feature Requirements**
   - Need Iceberg-exclusive capabilities not available in Delta UniForm
   - Advanced partition evolution or hidden partitioning patterns
   - Iceberg-native time travel or branch/tag management workflows

2. **External Ownership or Non-Databricks Primary Writer**
   - Another system (Snowflake, Flink, Spark outside Databricks) is the primary writer
   - Databricks is a consumer, not the authoritative source
   - Shared multi-vendor write access (though coordination is complex)
---

##9. Use DESCRIBE HISTORY together with the metadata columns to trace a specific bad row back to the exact ingestion run and source file that introduced it.

In [0]:
--check for bad data, once we find the details of bad data from here then we use describe history for traceback to source then we can use time trave to rollback the changes
SELECT *, _metadata.file_modification_time
FROM cyntexa_dev.sales.sales_raw_combined
WHERE discount_amount =0;
DESCRIBE HISTORY cyntexa_dev.sales.sales_raw_combined;